# Lab 1 - Esercizio 3.1: Miglioramento delle prestazioni del fine-tuning

Questo notebook documenta il passaggio dalla baseline head-only dell'Esercizio 1.3 a un fine-tuning piu' efficace. L'obiettivo non e' provare molte idee in modo casuale, ma confrontare poche varianti motivate e discutere cosa ci dicono i risultati.

I risultati riportati qui sono quelli degli artifact della pipeline corrente in `artifacts/runs/`; le conclusioni finali non usano run storiche.


---
### Esercizio 3.1 (facile): miglioramento delle prestazioni del fine-tuning

L'esercizio richiede di migliorare il fine-tuning dell'Esercizio 1.3 valutando alternative motivate: modello più capace, augmentation, sblocco selettivo dei layer o altre forme di regolarizzazione.

La scelta adottata è lo **sblocco selettivo di `layer4`**: le feature di alto livello vengono adattate a GTSRB senza aggiornare l'intera ResNet-18.


In [ ]:
from pathlib import Path
import subprocess

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

from dla_lab1.config import experiment_config, load_config
from dla_lab1.experiments import batch_size_for, run_finetuning
from dla_lab1.evaluate import history_to_frame
from dla_lab1.models import build_classifier, count_parameters

config = load_config(ROOT / "config" / "config.yaml")

# Non rilanciamo training lunghi automaticamente.
RUN_TRAINING = False

print(f"Radice del progetto: {ROOT.name}")


Project root: DLA_1


## Punto di partenza: baseline Esercizio 1.3

La baseline head-only e' il riferimento metodologico: ResNet-18 pre-addestrata, backbone congelato e solo classificatore finale addestrabile.

In questo notebook non copiamo valori numerici dal vecchio lavoro. Il confronto deve essere fatto con risultati prodotti dalla pipeline corrente o con artifact generati localmente.


In [2]:
baseline_context = pd.DataFrame([
    ["Esercizio 1.2", "ResNet-18 feature extractor + SVM", "baseline stabile senza fine-tuning"],
    ["Esercizio 1.3", "ResNet-18 frozen backbone + linear fc", "fine-tuning head-only"],
], columns=["Reference", "Setup", "Role"])

baseline_context


,Reference,Setup,Role
0,Exercise 1.2,ResNet-18 feature extractor + SVM,baseline stabile senza fine-tuning
1,Exercise 1.3,ResNet-18 frozen backbone + linear fc,fine-tuning head-only


Nota: i risultati prodotti in notebook vecchi possono aiutare a ricordare quali idee erano state provate, ma non devono essere usati come risultati finali in questa versione. Qui manteniamo solo il metodo e rilanciamo le configurazioni rilevanti nella pipeline corrente.


## Esperimenti possibili per Esercizio 3.1

Le idee da provare sono tre:

- sbloccare `layer4` mantenendo congelato il resto del backbone;
- aggiungere augmentation moderata;
- combinare `layer4` trainabile e augmentation.

La tabella seguente elenca le configurazioni disponibili, senza riportare risultati copiati dal vecchio notebook.


### Loss considerate nello screening

Gli screening sullo sbilanciamento hanno considerato weighted cross-entropy e Focal Loss. La prima assegna più peso alle classi rare usando conteggi inversi alla frequenza; `class_weights_from_labels` usa `np.bincount`, impone un conteggio minimo pari a uno e passa i pesi a `nn.CrossEntropyLoss` tramite `build_loss`.

La `FocalLoss` locale riduce invece il contributo degli esempi già classificati con sicurezza e concentra maggiormente l'ottimizzazione su quelli difficili. Nell'esperimento lo screening usa `gamma=2` e `alpha=1`. Il modello finale selezionato usa cross-entropy con label smoothing `0.05`, quindi queste varianti restano documentate a parole senza ripetere formule secondarie non presenti nel README definitivo.


In [3]:
improvement_candidates = pd.DataFrame([
    ["ex3_1_layer4_unfrozen", "layer4 + fc", "none", "verifica dell'effetto dello sblocco di layer4"],
    ["ex3_1_head_only_aggressive_aug", "fc only", "aggressive", "controllo: augmentation senza fine-tuning del backbone"],
    ["ex3_1_layer4_aggressive_aug", "layer4 + fc", "aggressive", "augmentation forte con layer4 trainabile"],
    ["ex3_1_layer4_conservative_aug", "layer4 + fc", "conservative", "augmentation realistica con layer4 trainabile"],
    ["ex3_1_layer4_spatial_aug", "layer4 + fc", "spatial", "sole perturbazioni geometriche leggere"],
    ["ex3_1_layer4_no_aug_lr1e4_wd05", "layer4 + fc", "none", "LR piu basso e weight decay piu alto senza augmentation"],
    ["ex3_1_layer4_safe_aug_ls005_discriminative", "layer4 + fc", "safe", "augmentation sicura, label smoothing e LR discriminativi"],
    ["ex3_1_layer4_conservative_ls005", "layer4 + fc", "conservative", "label smoothing su augmentation conservativa"],
], columns=["Experiment", "Trainable layers", "Augmentation", "Purpose"])

improvement_candidates


,Experiment,Trainable layers,Augmentation,Purpose
0,ex3_1_layer4_unfrozen,layer4 + fc,none,verifica dell'effetto dello sblocco di layer4
1,ex3_1_head_only_aggressive_aug,fc only,aggressive,controllo: augmentation senza fine-tuning del ...
2,ex3_1_layer4_aggressive_aug,layer4 + fc,aggressive,augmentation forte con layer4 trainabile
3,ex3_1_layer4_conservative_aug,layer4 + fc,conservative,augmentation realistica con layer4 trainabile
4,ex3_1_layer4_spatial_aug,layer4 + fc,spatial,sole perturbazioni geometriche leggere
5,ex3_1_layer4_no_aug_lr1e4_wd05,layer4 + fc,none,LR piu basso e weight decay piu alto senza aug...
6,ex3_1_layer4_safe_aug_ls005_discriminative,layer4 + fc,safe,"augmentation sicura, label smoothing e LR disc..."
7,ex3_1_layer4_conservative_ls005,layer4 + fc,conservative,label smoothing su augmentation conservativa


## Strategia sperimentale

La baseline head-only allena solo la `fc` finale. Se le feature ImageNet non separano bene i segnali GTSRB, questa scelta e' troppo rigida. Per questo le varianti dell'Esercizio 3.1 provano a sbloccare `layer4`, cioe' l'ultimo blocco convoluzionale della ResNet-18, lasciando congelati i blocchi precedenti.

Le augmentation vengono valutate con cautela. Per GTSRB alcune trasformazioni sono semanticamente rischiose: ad esempio il flip orizzontale puo' cambiare il significato di segnali direzionali. Per questo la variante aggressiva e' utile come esperimento, ma non e' automaticamente una buona scelta finale.


## Risultati della pipeline corrente

| Run | Parametri allenati | Augmentation | Best val acc | Ultima train acc | Ultima val acc | Gap train-val finale |
|---|---|---|---:|---:|---:|---:|
| `ex1_3_head_only_adam_ce` | `fc` | none | 0.4665 | 0.8272 | 0.4653 | 0.3619 |
| `ex3_1_head_only_aggressive_aug` | `fc` | aggressive | 0.4387 | 0.6678 | 0.4296 | 0.2383 |
| `ex3_1_layer4_aggressive_aug` | `layer4` + `fc` | aggressive | 0.7433 | 0.9927 | 0.7369 | 0.2557 |
| `ex3_1_layer4_conservative_aug` | `layer4` + `fc` | conservative | 0.7541 | 0.9979 | 0.7512 | 0.2467 |
| `ex3_1_layer4_spatial_aug` | `layer4` + `fc` | spatial | 0.7089 | 0.9986 | 0.7089 | 0.2897 |
| `ex3_1_layer4_unfrozen` | `layer4` + `fc` | none | 0.7584 | 1.0000 | 0.7584 | 0.2416 |
| `ex3_1_layer4_no_aug_lr1e4_wd05` | `layer4` + `fc` | none | 0.6947 | 1.0000 | 0.6938 | 0.3062 |
| `ex3_1_layer4_safe_aug_ls005_discriminative` | `layer4` + `fc` | safe | 0.7306 | 0.9985 | 0.7282 | 0.2703 |
| `ex3_1_layer4_conservative_ls005` | `layer4` + `fc` | conservative | **0.7741** | 0.9998 | 0.7701 | 0.2297 |

La configurazione migliore e' `ex3_1_layer4_conservative_ls005`. Il risultato mostra che, in questo setup, il contributo piu' efficace non e' l'augmentation aggressiva ma una regolarizzazione leggera della loss (`label_smoothing=0.05`) insieme a trasformazioni realistiche per i segnali stradali. La variante `safe` con learning rate discriminativi e' piu' prudente ma non raggiunge la stessa accuratezza; il solo abbassamento del learning rate senza augmentation peggiora sensibilmente la validazione.


In [4]:
candidate_runs = [
    "ex3_1_layer4_unfrozen",
    "ex3_1_layer4_conservative_aug",
    "ex3_1_layer4_spatial_aug",
    "ex3_1_layer4_no_aug_lr1e4_wd05",
    "ex3_1_layer4_safe_aug_ls005_discriminative",
    "ex3_1_layer4_conservative_ls005",
]

run_rows = []
for run_name in candidate_runs:
    summary_path = ROOT / "artifacts" / "runs" / run_name / "summary.json"
    if summary_path.exists():
        summary = pd.read_json(summary_path, typ="series")
        run_rows.append([
            run_name,
            summary.get("best_epoch", None),
            summary.get("best_val_acc", None),
            summary_path,
        ])

if run_rows:
    current_ex3_results = pd.DataFrame(
        run_rows,
        columns=["Experiment", "Best Epoch", "Best Val Accuracy", "Summary Path"],
    ).sort_values("Best Val Accuracy", ascending=False)
    display(current_ex3_results)
else:
    print("Nessun summary.json trovato per le run candidate.")


,Experiment,Best Epoch,Best Val Accuracy,Summary Path
5,ex3_1_layer4_conservative_ls005,20.0,0.768729,DLA_1/...
0,ex3_1_layer4_unfrozen,15.0,0.758419,DLA_1/...
1,ex3_1_layer4_conservative_aug,9.0,0.754124,DLA_1/...
4,ex3_1_layer4_safe_aug_ls005_discriminative,15.0,0.730584,DLA_1/...
2,ex3_1_layer4_spatial_aug,15.0,0.708935,DLA_1/...
3,ex3_1_layer4_no_aug_lr1e4_wd05,13.0,0.694674,DLA_1/...


## Configurazioni nella nuova pipeline

Le configurazioni utili per Esercizio 3.1 sono ora in `config/config.yaml`. Questo permette di rilanciare le prove senza riscrivere celle lunghe nel notebook.


In [5]:
exercise_31_experiments = [
    "ex3_1_layer4_unfrozen",
    "ex3_1_layer4_conservative_aug",
    "ex3_1_layer4_spatial_aug",
    "ex3_1_layer4_no_aug_lr1e4_wd05",
    "ex3_1_layer4_safe_aug_ls005_discriminative",
    "ex3_1_layer4_conservative_ls005",
]

config_rows = []
for name in exercise_31_experiments:
    exp = experiment_config(config, name)
    config_rows.append([
        name,
        exp["model"]["name"],
        exp["model"]["unfreeze_layer4"],
        exp["experiment"].get("augmentation", "none"),
        exp["training"]["optimizer"],
        exp["training"]["loss"],
        exp["training"].get("label_smoothing", 0.0),
        exp["training"]["learning_rate"],
        exp["training"].get("layer4_lr", None),
        exp["training"].get("fc_lr", None),
        exp["training"]["epochs"],
        batch_size_for(config, exp["experiment"]["batch_size_key"]),
    ])

pd.DataFrame(config_rows, columns=[
    "Experiment", "Model", "Unfreeze layer4", "Augmentation", "Optimizer",
    "Loss", "Label smoothing", "Learning rate", "Layer4 LR", "FC LR",
    "Epochs", "Batch size"
])


,Experiment,Model,Unfreeze layer4,Augmentation,Optimizer,Loss,Label smoothing,Learning rate,Layer4 LR,FC LR,Epochs,Batch size
0,ex3_1_layer4_unfrozen,resnet18,True,none,AdamW,CrossEntropy,0.00,0.0005,NaN,NaN,15,128
1,ex3_1_layer4_conservative_aug,resnet18,True,conservative,AdamW,CrossEntropy,0.00,0.0005,NaN,NaN,10,128
2,ex3_1_layer4_spatial_aug,resnet18,True,spatial,AdamW,CrossEntropy,0.00,0.0005,NaN,NaN,10,128
3,ex3_1_layer4_no_aug_lr1e4_wd05,resnet18,True,none,AdamW,CrossEntropy,0.00,0.0001,NaN,NaN,20,128
4,ex3_1_layer4_safe_aug_ls005_discriminative,resnet18,True,safe,AdamW,CrossEntropy,0.05,0.0005,0.0001,0.0005,20,128
5,ex3_1_layer4_conservative_ls005,resnet18,True,conservative,AdamW,CrossEntropy,0.05,0.0005,NaN,NaN,20,128


## Modello selezionato e overfitting

Il modello selezionato e' `ex3_1_layer4_conservative_ls005`, perche' ottiene la migliore validation accuracy tra le configurazioni confrontate: **0.7741** alla epoca 11. La scelta usa solo training e validation; il test set resta separato fino alla valutazione finale.

Il modello mostra ancora overfitting: la train accuracy finale e' **0.9998**, mentre la validation finale e' **0.7701**. Questo non indica data leakage. Indica che il blocco `layer4`, una volta sbloccato, ha capacita' sufficiente per adattarsi quasi perfettamente al training split. La regolarizzazione con `label_smoothing=0.05` migliora la validation e riduce leggermente il gap rispetto al precedente best, ma non annulla la differenza tra training e validation.

La causa principale e' la combinazione di tre fattori: GTSRB e' sbilanciato, molte immagini della stessa classe sono molto simili tra loro, e il fine-tuning di `layer4` aumenta molto la capacita' rispetto alla sola testa finale. Per questo motivo le trasformazioni aggressive non sono una soluzione appropriata: l'horizontal flip puo' cambiare il significato dei segnali direzionali, mentre variazioni cromatiche forti possono alterare colori semanticamente importanti.


In [6]:
SELECTED_EXPERIMENT = "ex3_1_layer4_conservative_ls005"
selected_cfg = experiment_config(config, SELECTED_EXPERIMENT)

model = build_classifier(
    model_name=selected_cfg["model"]["name"],
    num_classes=int(config["project"]["num_classes"]),
    weights=selected_cfg["model"].get("weights", "DEFAULT"),
    freeze_backbone=bool(selected_cfg["model"].get("freeze_backbone", True)),
    unfreeze_layer4=bool(selected_cfg["model"].get("unfreeze_layer4", False)),
)

parameter_summary = pd.DataFrame([count_parameters(model)])
parameter_summary["trainable_ratio"] = parameter_summary["trainable"] / parameter_summary["total"]
parameter_summary


,total,trainable,trainable_ratio
0,11198571,8415787,0.751505


Il numero di parametri trainabili e' molto piu' alto della baseline head-only, ma resta inferiore al fine-tuning completo. Questo e' il compromesso principale dell'esercizio: piu' adattamento al dataset senza aggiornare tutta la rete.


## Cella di riproducibilita' del miglior esperimento

La cella seguente e' disattivata di default. Serve solo se vuoi rieseguire la variante selezionata nella nuova pipeline.


In [7]:
if RUN_TRAINING:
    result = run_finetuning(config, SELECTED_EXPERIMENT, root=ROOT)
    history = history_to_frame(result["history"])
    display(history)
    print(f"Run salvata in: {result['artifacts']['output_dir']}")
else:
    print("Training non eseguito. Imposta RUN_TRAINING = True per rilanciare Esercizio 3.1.")


  0%|          | 0/163 [00:10<?, ?it/s]

  0%|          | 0/46 [00:15<?, ?it/s]

Epoch 01 | lr=5.00e-04 | train_loss=0.9688 train_acc=0.8197 | val_loss=1.5181 val_acc=0.6696


  0%|          | 0/163 [00:00<?, ?it/s]

  0%|          | 0/46 [00:00<?, ?it/s]

...

  0%|          | 0/163 [00:00<?, ?it/s]

  0%|          | 0/46 [00:00<?, ?it/s]

Epoch 17 | lr=6.25e-05 | train_loss=0.3825 train_acc=0.9999 | val_loss=1.1895 val_acc=0.7649
Early stopping at epoch 17.


,epoch,train_loss,train_acc,val_loss,val_acc,learning_rate
0,1,0.968836,0.819693,1.518142,0.669588,0.000500
1,2,0.525911,0.963497,1.404268,0.702749,0.000500
2,3,0.468455,0.980211,1.354886,0.709107,0.000500
3,4,0.438341,0.987608,1.316710,0.721134,0.000500
4,5,0.424302,0.990634,1.265309,0.731787,0.000500
5,6,0.401126,0.996590,1.230263,0.751718,0.000250
6,7,0.393494,0.998079,1.282353,0.742784,0.000250
7,8,0.390465,0.998703,1.249595,0.748969,0.000250
8,9,0.390165,0.998607,1.215152,0.757560,0.000250
9,10,0.389237,0.998895,1.204039,0.756873,0.000250


Run salvata in: DLA_1\artifacts\runs\ex3_1_layer4_conservative_ls005


## Valutazione finale sul test set del best model

Dopo la selezione su validation, il test set viene usato una sola volta per stimare la generalizzazione finale del modello scelto. Questa e' la procedura corretta: il test non partecipa alla scelta di augmentation, learning rate, label smoothing o checkpoint.

La cella e' disattivata di default per evitare riesecuzioni accidentali durante la lettura del notebook. Il risultato finale e' riferito al checkpoint `ex3_1_layer4_conservative_ls005.pt`.

La valutazione finale eseguita sul test set produce **accuracy = 0.8025**, **macro F1 = 0.74** e **weighted F1 = 0.80**. Questo risultato non e' stato usato per modificare iperparametri o scegliere un nuovo checkpoint: serve solo come stima finale della generalizzazione.


In [8]:
RUN_FINAL_TEST_EVAL = False
FINAL_EXPERIMENT = "ex3_1_layer4_conservative_ls005"

if RUN_FINAL_TEST_EVAL:
    result = subprocess.run(
        [sys.executable, "scripts/evaluate_checkpoint.py", "--experiment", FINAL_EXPERIMENT],
        cwd=ROOT,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Valutazione finale fallita con codice {result.returncode}")
else:
    print(
        "Test finale fissato sul checkpoint ex3_1_layer4_conservative_ls005.pt. "
        "Non usare il test per ulteriori scelte di iperparametri."
    )


Test accuracy Exercise 3.1: 0.8025
              precision    recall  f1-score   support

           0       0.72      0.35      0.47        60
           1       0.68      0.63      0.65       720
           2       0.62      0.73      0.67       750
           3       0.71      0.68      0.69       450
           4       0.74      0.83      0.78       660
           5       0.65      0.67      0.66       630
           6       0.90      0.93      0.91       150
           7       0.89      0.78      0.83       450
           8       0.72      0.74      0.73       450
           9       0.94      0.85      0.89       480
          10       0.91      0.99      0.95       660
          11       0.80      0.78      0.79       420
          12       1.00      0.99      0.99       690
          13       0.99      1.00      1.00       720
          14       0.94      0.97      0.96       270
          15       0.98      0.99      0.99       210
          16       1.00      0.92      0.96   

## Funzioni usate

Le funzioni principali sono concentrate nel package `src/dla_lab1`, in modo che il notebook resti leggibile e la pipeline sia riutilizzabile.

- `build_transforms` in `transforms.py` definisce preprocessing e augmentation (`none`, `aggressive`, `conservative`, `safe`, `spatial`).
- `build_classifier` in `models.py` costruisce ResNet-18 e applica la scelta di congelare o sbloccare `layer4`.
- `train_model` in `train.py` esegue il ciclo di training, salva il checkpoint migliore e applica early stopping.
- `trainable_parameter_groups` in `train.py` permette learning rate separati per `layer4` e `fc` quando richiesto dal config.
- `build_loss` in `losses.py` costruisce la loss e applica `label_smoothing` per la run finale.
- `run_finetuning` in `experiments.py` collega configurazione, dataloader, modello, training e salvataggio degli artifact.


## Conclusione Esercizio 3.1

La progressione sperimentale e' coerente: la baseline head-only arriva a circa **46,6%** di validation accuracy, mentre il fine-tuning selettivo di `layer4` porta il modello oltre il **75%**. La migliore configurazione finale e' `ex3_1_layer4_conservative_ls005`, con **77,4%** di validation accuracy.

La scelta finale usa augmentation conservativa e `label_smoothing=0.05`. Questa combinazione e' adatta al dominio: introduce piccole variazioni realistiche di crop, rotazione, luminosita' e contrasto, senza trasformazioni semanticamente pericolose come horizontal flip o hue shift marcati. Il risultato e' superiore sia alla run senza augmentation sia alle varianti aggressive.

Il comportamento di overfitting resta visibile e viene riportato esplicitamente: la train accuracy quasi perfetta mostra che il modello si adatta molto bene al training set, mentre la validation misura la difficolta' di generalizzare su immagini non viste. La conclusione sperimentale e' quindi precisa: il fine-tuning di `layer4` e la regolarizzazione leggera migliorano la generalizzazione, ma il task rimane sensibile alla capacita' del modello e allo sbilanciamento delle classi.

Sul test set, valutato una sola volta dopo la selezione su validation, il checkpoint finale raggiunge **80,25%** di accuracy. La macro F1 pari a **0.74** e' piu' bassa della weighted F1 pari a **0.80**, quindi le classi rare restano piu' difficili rispetto alle classi frequenti. Questo e' coerente con l'analisi esplorativa iniziale, che aveva evidenziato uno sbilanciamento marcato nella distribuzione delle classi.

La procedura train/validation/test e' corretta. Train aggiorna i pesi, validation seleziona modello e iperparametri, test fornisce solo la stima finale dopo la scelta. In questo modo non c'e' leakage metodologico dal test set.


## Funzioni, classi e moduli locali richiamati

La tabella è ricavata dagli import, dagli utilizzi nelle celle e dalle definizioni effettive nei package locali.

| Funzione o classe | Tipo | File di definizione | Scopo | Input principali | Output principali | Sezione |
| ----------------- | ---- | ------------------- | ----- | ---------------- | ----------------- | ------- |
| `batch_size_for` | Funzione | `src/dla_lab1/experiments.py` | Serve a leggere il batch size dalla sezione hardware del config. | `config: dict`, `batch_size_key: str` | Batch size intero da usare nella run. | Configurazioni nella nuova pipeline |
| `build_classifier` | Funzione | `src/dla_lab1/models.py` | Serve a creare un modello pre-addestrato adattato alle 43 classi GTSRB. | `model_name: str`, `num_classes: int`, `weights: str \| None`, `freeze_backbone: bool`, `unfreeze_layer4: bool` | Modello PyTorch con layer finale sostituito e parametri addestrabili configurati. | Modello selezionato e overfitting |
| `count_parameters` | Funzione | `src/dla_lab1/models.py` | Serve a contare parametri totali e addestrabili. | `model: nn.Module` | Dizionario con `total` e `trainable`. | Modello selezionato e overfitting |
| `experiment_config` | Funzione | `src/dla_lab1/config.py` | Serve a costruire la configurazione completa di un esperimento. | `config: dict[str, Any]`, `name: str` | Configurazione completa della run, con model/training gia' risolti. | Configurazioni nella nuova pipeline; Modello selezionato e overfitting |
| `history_to_frame` | Funzione | `src/dla_lab1/evaluate.py` | Serve a trasformare la storia del training in una tabella Pandas. | `history` | DataFrame con una riga per epoca. | Cella di riproducibilita' del miglior esperimento |
| `load_config` | Funzione | `src/dla_lab1/config.py` | Serve a leggere il file YAML di configurazione. | `path: str \| Path` | Dizionario Python con tutte le sezioni della configurazione. | Esercizio 3.1 (facile): miglioramento delle prestazioni del fine-tuning |
| `run_finetuning` | Funzione | `src/dla_lab1/experiments.py` | Serve a lanciare un esperimento di fine-tuning definito nel config. | `config: dict`, `experiment_name: str`, `root: str \| Path` | Dizionario con modello addestrato, history, DataLoader, device e artifact salvati. | Cella di riproducibilita' del miglior esperimento |
